In [ ]:
import os
import csv
import warnings
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats
from scipy.stats import pearsonr
import ukbiobank.utils.utils
from ukbiobank.utils import loadCsv
from ukbiobank.utils import addFields
from ukbiobank.utils.utils import fieldIdsToNames

In [ ]:
# Upload UK Bioabank csv
csv_path = '/UK_BB/ukbbdata/ukb.csv'
ukb = ukbiobank.ukbio(ukb_csv=csv_path)

In [ ]:
# Upload required fields
df_demo = ukbiobank.utils.utils.loadCsv(ukbio=ukb, fields=['eid',
31, #Sex
21003, #Age when attended assessment centre
21000 #Ethnic background
])
df_demo.to_csv('/UK_BB/brainbody/demographics_full.csv', index=False)

## Full cognitive performance sample

In [ ]:
# Combine data across folds: cognitive data full sample
folds = range(0, 5)
directory = r'Z:\UK_BB\brainbody\cognition\folds'

# Create empty lists to store all data
all_train_data = []
all_test_data = []

for fold in folds:
    # Process training data
    g_train = pd.read_csv(os.path.join(directory, f'fold_{fold}', 'g', f'g_train_with_id_{fold}.csv'))
    g_train['fold'] = fold  # Add fold identifier
    g_train['split'] = 'train'  # Add split identifier
    print(g_train.shape)
    
    # Process test data
    g_test = pd.read_csv(os.path.join(directory, f'fold_{fold}', 'g', f'g_test_with_id_{fold}.csv'))
    g_test['fold'] = fold  # Add fold identifier
    g_test['split'] = 'test'  # Add split identifier
    print(g_test.shape)
    all_test_data.append(g_test)

# Combine all folds into single tables
all_test_combined_full_cog = pd.concat(all_test_data, ignore_index=True)
print(all_test_combined_full_cog.shape)

(25517, 4)
(6380, 4)
(25517, 4)
(6380, 4)
(25518, 4)
(6379, 4)
(25518, 4)
(6379, 4)
(25518, 4)
(6379, 4)
(31897, 4)


Rename columns and count NAs

In [ ]:
# Rename columns and count NAs
df_demo = pd.read_csv(r'Z:\UK_BB\brainbody\demographics_full.csv')
df_demo_i2 = df_demo[[
'eid',
'31-0.0',
'21000-0.0',
'21003-2.0',]]
df_demo_i2 = df_demo_i2.rename(columns={
'31-0.0':'Sex',
'21000-0.0':'Ethnicity',
'21003-2.0':'Age',
})
demo_full_cog = pd.merge(all_test_combined_full_cog,df_demo_i2, on = 'eid')
print('NA:', demo_full_cog.isna().sum())

NA: g            0
eid          0
fold         0
split        0
Sex          0
Ethnicity    7
Age          0
dtype: int64


Display age and sex

In [5]:
print('Sample size', demo_full_cog.shape[0])
print('Mean age', demo_full_cog['Age'].mean().round(2))
print(f"SD age {demo_full_cog['Age'].std():.3f}")
print('Age max range:', demo_full_cog['Age'].max())
print('Age min range:', demo_full_cog['Age'].min())
print('Proportion of males:', (demo_full_cog['Sex'].value_counts()[1] / len(demo_full_cog['Sex']) * 100).round(2))
print('Proportion of females:', (demo_full_cog['Sex'].value_counts()[0] / len(demo_full_cog['Sex']) * 100).round(2))

Sample size 31897
Mean age 64.55
SD age 7.663
Age max range: 83.0
Age min range: 46.0
Proportion of males: 48.68
Proportion of females: 51.32


Ethnicity

In [6]:
# Count ethnicity
demo_full_cog_ethnicity = demo_full_cog.dropna()
# Get counts
counts = demo_full_cog_ethnicity['Ethnicity'].value_counts()
# Calculate percentages
percentages = demo_full_cog_ethnicity['Ethnicity'].value_counts(normalize=True).mul(100).round(2)

# Combine into one DataFrame
result = pd.DataFrame({
    'Count': counts,
    'Percentage': percentages
}).reset_index()

# Create a mapping dictionary
ethnicity_mapping = {
1001.0:'British',
1003.0: 'Any other white background',
1002.0: 'Irish'
}
# Apply the mapping
result['Ethnicity'] = result['Ethnicity'].replace(ethnicity_mapping)
white_categories = ['British', 'Irish', 'Any other white background']
percent_white = result.loc[result['Ethnicity'].isin(white_categories), 'Percentage'].sum()
print(result)
print('% White:', percent_white)

                     Ethnicity  Count  Percentage
0                      British  29120       91.31
1   Any other white background   1054        3.31
2                        Irish    795        2.49
3                       3001.0    203        0.64
4                          6.0    150        0.47
5                       4001.0    101        0.32
6                          5.0     80        0.25
7                       4002.0     67        0.21
8                         -3.0     65        0.20
9                       2003.0     49        0.15
10                      3004.0     48        0.15
11                      2004.0     48        0.15
12                      3002.0     33        0.10
13                      2001.0     30        0.09
14                      2002.0     20        0.06
15                         1.0     13        0.04
16                      3003.0      5        0.02
17                        -1.0      4        0.01
18                      4003.0      3        0.01


## Commonality analysis sample

In [ ]:
# Combine data across folds
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

folds = range(0, 5)
commonality_path = r'Z:\UK_BB\brainbody\commonality'
folds = range(0, 5)
stacking_path = r'Z:\UK_BB\brainbody\stacking'
base_path = r'Z:\UK_BB\brainbody'
behav_mod = 'activity_byrecall_i3'

modalities_to_compare = {
    'allmri': ('brain', 'allmri'),
    'dwi': ('brain', 'dwi'),
    'rs': ('brain', 'rs'),
    'smri': ('brain', 'smri'),
    'body': ('body', None),
    'brain-body': ('brain-body', None)
}

df_concat = []
for fold in folds:
    # Load behavioural domain predictio
    behav_pred_path = os.path.join(
    base_path,
    'lifestyle-envir-body',
    'folds', f'fold_{fold}', 'g_pred',
    f'{behav_mod}_g_pred_XGB_test_with_id_fold_{fold}.csv')
            
    behav_pred = pd.read_csv(behav_pred_path).rename(
                columns={f'g_pred_test_{behav_mod}': f'g_pred_behav_{behav_mod}_test'})
            
    comp_path = os.path.join(
         stacking_path, 'brain', 'allmri', 'folds', f'fold_{fold}', 'g_pred',
                    f'allmri_target_pred_2nd_level_rf_test_fold_{fold}.csv')
            
    comp_pred = pd.read_csv(comp_path).rename(
                columns={'g_pred_stack_test': f'g_pred_allmri_test'})
            
    # Load observed g for this behavioural domain
    obs_path = os.path.join(
                stacking_path,
                'lifestyle-envir',
                'features_test_level1_stacked_outer',
                f'features_test_level1_outer_g_matched_fold_{fold}.csv')
            
    g_obs = pd.read_csv(obs_path)[['eid', 'g']].rename(columns={'g': 'g_obs_test'})
            
    # Merge
    merged = (
    behav_pred
    .merge(comp_pred, on='eid')
    .merge(g_obs, on='eid'))
            
    df_concat.append(merged)
        
    # Combine folds
    all_g = pd.concat(df_concat, axis=0, ignore_index=True).dropna()
        
    # Save merged dataset
    output_name = f'g_obs_pred_{behav_mod}_vs_allmri_with_id.csv'
    all_g.to_csv(os.path.join(commonality_path, output_name), index=False)

In [8]:
# Count NAs
sample_size_commonality = pd.read_csv(os.path.join(commonality_path, f'g_obs_pred_{behav_mod}_vs_allmri_with_id.csv'))
demo_commonality = pd.merge(sample_size_commonality,df_demo_i2, on = 'eid')
print('NA:', demo_full_cog.isna().sum())

NA: g            0
eid          0
fold         0
split        0
Sex          0
Ethnicity    7
Age          0
dtype: int64


Display age and sex

In [9]:
print('Sample size', demo_commonality.shape[0])
print('Mean age', demo_commonality['Age'].mean().round(2))
print(f"SD age {demo_commonality['Age'].std():.3f}")
print('Age max range:', demo_commonality['Age'].max())
print('Age min range:', demo_commonality['Age'].min())
print('Proportion of males:', (demo_commonality['Sex'].value_counts()[1] / len(demo_commonality['Sex']) * 100).round(2))
print('Proportion of females:', (demo_commonality['Sex'].value_counts()[0] / len(demo_commonality['Sex']) * 100).round(2))

Sample size 10133
Mean age 64.35
SD age 7.541
Age max range: 82.0
Age min range: 48.0
Proportion of males: 46.03
Proportion of females: 53.97


Ethnicity

In [10]:
# Count ethnicity
demo_commonality_ethnicity = demo_commonality.dropna()
# Get counts
counts = demo_commonality_ethnicity['Ethnicity'].value_counts()
# Calculate percentages
percentages = demo_commonality_ethnicity['Ethnicity'].value_counts(normalize=True).mul(100).round(2)

# Combine into one DataFrame
result = pd.DataFrame({
    'Count': counts,
    'Percentage': percentages
}).reset_index()

# Create a mapping dictionary
ethnicity_mapping = {
1001.0:'British',
1003.0: 'Any other white background',
1002.0: 'Irish'
}
# Apply the mapping to the Ethnicity_Code column
result['Ethnicity'] = result['Ethnicity'].replace(ethnicity_mapping)
white_categories = ['British', 'Irish', 'Any other white background']
percent_white = result.loc[result['Ethnicity'].isin(white_categories), 'Percentage'].sum()
print(result)
print('% White:', percent_white)

                     Ethnicity  Count  Percentage
0                      British   9219       91.02
1   Any other white background    389        3.84
2                        Irish    233        2.30
3                          6.0     59        0.58
4                       3001.0     58        0.57
5                       4001.0     34        0.34
6                          5.0     21        0.21
7                       2003.0     19        0.19
8                       3004.0     17        0.17
9                         -3.0     17        0.17
10                      2004.0     16        0.16
11                      4002.0     15        0.15
12                      2001.0     11        0.11
13                      2002.0      7        0.07
14                      3002.0      6        0.06
15                         1.0      5        0.05
16                      3003.0      2        0.02
17                         2.0      1        0.01
% White: 97.16
